<a href="https://colab.research.google.com/github/thansihatech/CSA0816-PYTHON/blob/main/ASSIGNMENT_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import os
import re

INPUT_FILES = [
    "CSE.txt",
    "ECE.txt",
    "IT.txt",
    "MECH.txt",
    "CIVIL.txt"
]

INPUT_DIR = "input"
OUTPUT_DIR = "output"


def validate_email(email):
    email = email.strip().lower()

    return (
        email.count("@") == 1
        and " " not in email
        and email.split("@")[0] != ""
        and "." in email.split("@")[1]
        and not email.split("@")[1].startswith(".")
        and not email.split("@")[1].endswith(".")
    )


def process_files():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR + "/events", exist_ok=True)

    seen = set()
    master = []
    event_lists = {}
    errors = []

    for filename in INPUT_FILES:
        path = os.path.join(INPUT_DIR, filename)

        try:
            with open(path, "r", encoding="utf-8") as file:
                for line_no, line in enumerate(file, 1):

                    try:
                        fields = line.strip().split("|")

                        if len(fields) != 5:
                            raise ValueError("Damaged line")

                        reg, name, dept, email, events = fields

                        reg = reg.strip()
                        name = name.strip()
                        dept = dept.strip()
                        email = email.strip().lower()

                        if not all([reg, name, dept, events]):
                            raise ValueError("Missing field")

                        if not validate_email(email):
                            raise ValueError("Invalid email")

                        if reg in seen:
                            raise ValueError("Duplicate registration")

                        seen.add(reg)

                        event_names = [
                            e.strip() for e in events.split(";")
                            if e.strip()
                        ]

                        student = (
                            f"{reg}|{name}|{dept}|{email}|"
                            f"{';'.join(event_names)}"
                        )

                        master.append(student)

                        for event in event_names:
                            event_lists.setdefault(event, []).append(
                                f"{reg}|{name}|{dept}|{email}"
                            )

                    except ValueError as error:
                        errors.append(
                            f"{filename}, Line {line_no}: {error}"
                        )

        except FileNotFoundError:
            errors.append(f"Missing file: {filename}")

        except OSError as error:
            errors.append(f"{filename}: File error - {error}")

    # Write master list
    with open(OUTPUT_DIR + "/master_list.txt",
              "w", encoding="utf-8") as file:
        file.write("\n".join(master))

    # Write event-wise files
    for event, students in event_lists.items():
        safe_name = re.sub(r"[^a-zA-Z0-9_-]", "_", event)

        with open(
            OUTPUT_DIR + "/events/" + safe_name + ".txt",
            "w", encoding="utf-8"
        ) as file:
            file.write("\n".join(students))

    # Write summary report
    with open(OUTPUT_DIR + "/summary.txt",
              "w", encoding="utf-8") as file:
        file.write("COLLEGE FEST SUMMARY\n")
        file.write(f"Total valid registrations: {len(master)}\n")
        file.write(f"Total events: {len(event_lists)}\n")
        file.write(f"Total errors: {len(errors)}\n")

        for event, students in event_lists.items():
            file.write(f"{event}: {len(students)} participants\n")

    # Write error log
    with open(OUTPUT_DIR + "/error_log.txt",
              "w", encoding="utf-8") as file:
        file.write("\n".join(errors))

    print("Processing completed!")
    print("Valid registrations:", len(master))
    print("Errors:", len(errors))
    print("Output files created successfully.")


process_files()


Processing completed!
Valid registrations: 0
Errors: 5
Output files created successfully.
